# Organoid Regulome Benchmark: hgx on Fleck et al. 2023
GPU-accelerated hypergraph deep learning on cerebral organoid GRNs.
Requires Colab Pro/Pro+ with **A100 GPU** runtime.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, subprocess
DRIVE = '/content/drive/MyDrive/organoid-hgx-benchmark'
DATA = f'{DRIVE}/data'
FIG = f'{DRIVE}/figures'
os.makedirs(FIG, exist_ok=True)

!pip install -q jax[cuda12] equinox diffrax optax jaxtyping scanpy anndata ripser matplotlib networkx scikit-learn

for repo, url in [('hgx','https://github.com/m9h/hgx'),
                   ('devograph','https://github.com/m9h/devograph'),
                   ('organoid-hgx-benchmark','https://github.com/m9h/organoid-hgx-benchmark')]:
    p = f'/content/{repo}'
    if os.path.exists(p):
        !cd {p} && git pull -q
    else:
        !git clone -q {url} {p}
    if os.path.exists(f'{p}/pyproject.toml'):
        !pip install -q -e {p}

import jax; print(f'JAX {jax.__version__} on {jax.devices()}')
import hgx, devograph; print('hgx + devograph OK')

In [ ]:
files = {
    'pando/coefs.tsv': 'https://zenodo.org/records/15371701/files/grn_modules.tsv?download=1',
    'expression/RNA_data.h5ad': 'https://zenodo.org/records/15371701/files/RNA_data.h5ad?download=1',
    'zenodo/rna_matrices.tar.gz': 'https://zenodo.org/records/15371701/files/rna_matrices.tar.gz?download=1',
    'zenodo/RNA_all_velo.h5ad': 'https://zenodo.org/records/15371701/files/RNA_all_velo.h5ad?download=1',
}
for path, url in files.items():
    fp = f'{DATA}/{path}'
    os.makedirs(os.path.dirname(fp), exist_ok=True)
    if os.path.exists(fp):
        print(f'  CACHED: {path} ({os.path.getsize(fp)/1e6:.0f} MB)')
    else:
        print(f'  Downloading {path}...')
        subprocess.run(['curl', '-L', '-o', fp, url], check=True)
        print(f'    Done: {os.path.getsize(fp)/1e6:.0f} MB')
if not os.path.exists(f'{DATA}/zenodo/data_matrices'):
    !tar xzf {DATA}/zenodo/rna_matrices.tar.gz -C {DATA}/zenodo/
print('All data ready')

In [ ]:
work = '/content/organoid-hgx-benchmark'
for name, target in [('data', DATA), ('figures', FIG)]:
    link = f'{work}/{name}'
    if os.path.islink(link): os.unlink(link)
    elif os.path.isdir(link): shutil.rmtree(link)
    os.symlink(target, link)
# PPCA consensus dimensionality (default triggers auto-estimation)
!cd {work} && python scripts/00_preprocess.py

In [ ]:
import numpy as np, json, matplotlib.pyplot as plt
from pathlib import Path
import jax, jax.numpy as jnp, equinox as eqx, optax, hgx
from sklearn.decomposition import PCA

D = Path(f'{work}/data/processed')
inc = jnp.array(np.load(D/'incidence.npy'))
feat = jnp.array(np.load(D/'node_features_pca.npy'))
temp = np.load(D/'temporal_expression.npy')
pt = np.load(D/'pseudotime_centers.npy')
lf = np.load(D/'lineage_fractions.npy')
fp = np.load(D/'fate_probabilities.npy')
ml = np.load(D/'module_labels.npy')
eigvals = np.load(D/'eigenvalues.npy')
masks = jnp.array(np.load(D/'perturbation_masks.npy'))
effects = jnp.array(np.load(D/'perturbation_effects.npy'))
fates = jnp.array(np.load(D/'perturbation_fates.npy'))
with open(D/'gene_names.json') as f: genes = json.load(f)
with open(D/'tf_names.json') as f: tfs = json.load(f)
with open(D/'key_tf_indices.json') as f: kti = {k:int(v) for k,v in json.load(f).items()}
with open(D/'tf_gene_indices.json') as f: tgi = {k:int(v) for k,v in json.load(f).items()}

n_genes, n_edges = inc.shape
fd = feat.shape[1]
nc = int(ml.max()) + 1
hg = hgx.from_incidence(inc, node_features=feat)
key = jax.random.PRNGKey(42)
FIGP = Path(FIG)
C8 = ['#e41a1c','#377eb8','#4daf4a','#984ea3','#ff7f00','#a65628','#f781bf','#999999']
print(f'Loaded: {n_genes} genes, {n_edges} regulons, {fd}-dim features (PPCA consensus)')

In [ ]:
# FIGURE 1: GRN Architecture + TF Centrality
import networkx as nx
degrees = np.array(hg.node_degrees)
adj = np.array(hgx.clique_expansion(hg))
_, evecs = np.linalg.eigh(adj)
ec = np.abs(evecs[:,-1]); ec /= ec.max()+1e-8
bw = nx.betweenness_centrality(nx.from_numpy_array((adj>0).astype(float)), k=min(50,n_genes))
ba = np.array([bw.get(i,0) for i in range(n_genes)])
ed = np.array(hg.edge_degrees)

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes[0,0].hist(degrees, bins=50, color='#2c3e50', alpha=0.7)
for tf,idx in list(kti.items())[:4]:
    axes[0,0].axvline(x=degrees[idx], color='red', alpha=0.7, ls='--')
    axes[0,0].text(degrees[idx]+1, axes[0,0].get_ylim()[1]*0.85, tf, fontsize=8, rotation=45)
axes[0,0].set(xlabel='Node Degree', ylabel='Count', title=f'A: Degree Distribution ({n_genes} genes)')

x = np.arange(len(kti))
ti = [kti[t] for t in kti]
axes[0,1].bar(x-0.2,[degrees[i]/degrees.max() for i in ti],0.2,label='Degree',color='#3498db')
axes[0,1].bar(x,[ec[i] for i in ti],0.2,label='Eigenvector',color='#e74c3c')
axes[0,1].bar(x+0.2,[ba[i]/max(ba.max(),1e-8) for i in ti],0.2,label='Betweenness',color='#2ecc71')
axes[0,1].set_xticks(x); axes[0,1].set_xticklabels(list(kti.keys()), rotation=45, ha='right')
axes[0,1].set(ylabel='Centrality (norm)', title='B: Key TF Centrality'); axes[0,1].legend(fontsize=8)

axes[1,0].imshow(np.array(inc[:200,:30]).T, aspect='auto', cmap='Blues')
axes[1,0].set(xlabel='Gene (first 200)', ylabel='Regulon (first 30)', title=f'C: Incidence ({n_genes}x{n_edges})')

axes[1,1].hist(ed, bins=50, color='#8e44ad', alpha=0.7)
axes[1,1].set(xlabel='Regulon Size', ylabel='Count', title=f'D: Regulon Sizes (mean={ed.mean():.0f})')

fig.suptitle('Figure 1: GRN Architecture (Fleck et al. 2023)', fontsize=14, fontweight='bold')
fig.tight_layout(rect=[0,0,1,0.96])
fig.savefig(FIGP/'figure_01_grn.png', dpi=150, bbox_inches='tight'); plt.show(); print('Fig 1 done')

In [ ]:
# FIGURE 2: Module Detection (SheafDiffusion)
k1,k2 = jax.random.split(key)
nnz = int(jnp.sum(inc>0))
labels = jnp.array(ml)

class SM(eqx.Module):
    s: hgx.SheafDiffusion
    r: eqx.nn.Linear
    def __call__(self,hg): return jax.vmap(self.r)(self.s(hg))

mdl = SM(s=hgx.SheafDiffusion(num_steps=3,in_dim=fd,edge_stalk_dim=fd,num_incidences=nnz,key=k1),
         r=eqx.nn.Linear(fd,nc,key=k2))
opt=optax.adam(1e-3); ost=opt.init(eqx.filter(mdl,eqx.is_array))

@eqx.filter_jit
def stp(m,o,hg,lb):
    def lf(m,hg,lb):
        return -jnp.mean(jnp.sum(jax.nn.one_hot(lb,nc)*jax.nn.log_softmax(m(hg),-1),-1))
    l,g=eqx.filter_value_and_grad(lf)(m,hg,lb)
    u,no=opt.update(g,o,m); return eqx.apply_updates(m,u),no,l

losses=[]
for ep in range(200):
    mdl,ost,l=stp(mdl,ost,hg,labels); losses.append(float(l))
    if (ep+1)%50==0:
        p=jnp.argmax(mdl(hg),-1); print(f'  Ep {ep+1}: loss={l:.4f} acc={float(jnp.mean(p==labels)):.1%}')

preds=np.array(jnp.argmax(mdl(hg),-1))
fig,axes=plt.subplots(1,2,figsize=(14,5))
axes[0].plot(losses,lw=1,color='#2c3e50'); axes[0].set_yscale('log')
axes[0].set(xlabel='Epoch',ylabel='Loss',title=f'A: SheafDiffusion ({fd}-dim, acc={float(jnp.mean(jnp.array(preds)==labels)):.1%})')
axes[0].grid(True,alpha=0.3)
ms=sorted([(m,int((ml==m).sum())) for m in range(nc)],key=lambda x:-x[1])[:20]
ma=[float((preds[ml==m]==m).mean()) for m,_ in ms]
mn=[f'{tfs[m] if m<len(tfs) else m}\n({s})' for m,s in ms]
axes[1].bar(range(len(ma)),ma,color='steelblue')
axes[1].set_xticks(range(len(ma))); axes[1].set_xticklabels(mn,rotation=90,fontsize=6)
axes[1].set(ylabel='Accuracy',title='B: Per-Module (top 20)'); axes[1].set_ylim(0,1.1)
fig.suptitle('Figure 2: Module Detection',fontsize=14,fontweight='bold')
fig.tight_layout(rect=[0,0,1,0.96])
fig.savefig(FIGP/'figure_02_modules.png',dpi=150,bbox_inches='tight'); plt.show(); print('Fig 2 done')

In [ ]:
# FIGURE 3: Trajectory + Fate + PCA
fig,axes=plt.subplots(2,2,figsize=(14,12))
for i,(tf,idx) in enumerate(kti.items()):
    axes[0,0].plot(pt,temp[:,idx],'o-',color=C8[i],label=tf,lw=2,ms=4)
axes[0,0].set(xlabel='Pseudotime',ylabel='Expression (z)',title='A: Key TF Expression')
axes[0,0].legend(fontsize=7,ncol=2); axes[0,0].grid(True,alpha=0.3)

axes[0,1].stackplot(pt,lf[:,0],lf[:,1],lf[:,2],labels=['Telencephalon','Early','NT'],
                    colors=['#e41a1c','#377eb8','#4daf4a'],alpha=0.7)
axes[0,1].set(xlabel='Pseudotime',ylabel='Fraction',title='B: Lineage Composition')
axes[0,1].legend(fontsize=8); axes[0,1].set_ylim(0,1)

axes[1,0].plot(pt,fp[:,0],'o-',color='#e41a1c',label='DF (cortical)',lw=2)
axes[1,0].plot(pt,fp[:,1],'s-',color='#377eb8',label='VF (GE)',lw=2)
axes[1,0].plot(pt,fp[:,2],'^-',color='#4daf4a',label='MH (NT)',lw=2)
axes[1,0].set(xlabel='Pseudotime',ylabel='Fate Prob',title='C: CellRank Fates')
axes[1,0].legend(); axes[1,0].grid(True,alpha=0.3)

pc=PCA(2).fit_transform(np.array(feat))
axes[1,1].scatter(pc[:,0],pc[:,1],c=ml,cmap='tab20',s=3,alpha=0.5)
axes[1,1].set(xlabel='PC1',ylabel='PC2',title=f'D: Gene PCA ({fd}-dim, by module)')

fig.suptitle('Figure 3: Developmental Trajectory',fontsize=14,fontweight='bold')
fig.tight_layout(rect=[0,0,1,0.96])
fig.savefig(FIGP/'figure_03_trajectory.png',dpi=150,bbox_inches='tight'); plt.show(); print('Fig 3 done')

In [ ]:
# FIGURE 4: PPCA Eigenspectrum
fig,axes=plt.subplots(1,2,figsize=(14,5))
axes[0].plot(range(1,min(201,len(eigvals)+1)),eigvals[:200],'o-',ms=2,color='#2c3e50')
axes[0].axvline(x=26,color='red',ls='--',alpha=0.7,label='BIC=26')
axes[0].axvline(x=fd,color='blue',ls='--',alpha=0.7,label=f'Consensus={fd}')
axes[0].axvline(x=168,color='green',ls='--',alpha=0.7,label='AIC=168')
axes[0].set(xlabel='Component',ylabel='Eigenvalue',title='A: Scree Plot')
axes[0].legend(); axes[0].grid(True,alpha=0.3)
cv=np.cumsum(eigvals)/eigvals.sum()
axes[1].plot(range(1,len(cv)+1),cv,color='#8e44ad',lw=2)
axes[1].axhline(y=0.9,color='red',ls=':',alpha=0.5,label='90%')
axes[1].axhline(y=0.95,color='orange',ls=':',alpha=0.5,label='95%')
axes[1].axvline(x=fd,color='blue',ls='--',alpha=0.7,label=f'k={fd} ({cv[fd-1]:.1%})')
axes[1].set(xlabel='Components',ylabel='Cumulative Var',title='B: Variance Explained')
axes[1].legend(); axes[1].grid(True,alpha=0.3); axes[1].set_xlim(0,300)
fig.suptitle('Figure 4: PPCA Dimensionality (Minka/MELODIC)',fontsize=14,fontweight='bold')
fig.tight_layout(rect=[0,0,1,0.96])
fig.savefig(FIGP/'figure_04_eigenspectrum.png',dpi=150,bbox_inches='tight'); plt.show(); print('Fig 4 done')

In [ ]:
# FIGURE 5: Spectral Analysis
lap=np.array(hgx.hypergraph_laplacian(hg,normalized=True))
ev=np.linalg.eigvalsh(lap); nz=int((np.abs(ev)<1e-6).sum())
fig,axes=plt.subplots(1,2,figsize=(14,5))
axes[0].hist(ev[ev>1e-6],bins=80,color='#2c3e50',alpha=0.7)
axes[0].set(xlabel='Eigenvalue',ylabel='Count',title=f'A: Laplacian Spectrum (b0={nz})')
axes[1].plot(sorted(ev),'o',ms=2,color='#e74c3c')
axes[1].set(xlabel='Index',ylabel='Eigenvalue',title='B: Sorted Eigenvalues'); axes[1].grid(True,alpha=0.3)
fig.suptitle('Figure 5: Spectral Analysis of GRN Hypergraph',fontsize=14,fontweight='bold')
fig.tight_layout(rect=[0,0,1,0.96])
fig.savefig(FIGP/'figure_05_spectral.png',dpi=150,bbox_inches='tight'); plt.show(); print('Fig 5 done')

In [ ]:
# FIGURE 6: Neural ODE + SDE
import devograph, diffrax
T,ng=temp.shape; times=jnp.array(pt)
snaps=[hgx.from_incidence(inc,node_features=jnp.array(temp[t]).reshape(-1,1)) for t in range(T)]
thg=devograph.from_snapshots(snaps,times=times)
k1,k2,k3,k4=jax.random.split(key,4)

print('Training Neural ODE...')
ode=devograph.fit_neural_ode(thg,hgx.UniGCNConv(1,1,key=k1),key=k2,epochs=200,lr=1e-3)
pr=temp[0].reshape(-1,1); op=[temp[0]]
for t in range(T-1):
    pr=devograph.evolve(ode,hgx.from_incidence(inc,node_features=jnp.array(pr)),
                        t0=float(times[t]),t1=float(times[t+1])).node_features
    op.append(np.array(pr).flatten())
op=np.array(op); mse=float(np.mean((op[1:]-temp[1:])**2))
print(f'  ODE MSE: {mse:.6f}')

print('Training SDE...')
sde=devograph.HypergraphNeuralSDE(conv=hgx.UniGCNConv(1,1,key=k3),num_nodes=ng,node_dim=1,sigma_init=0.1,dt=0.05,key=k4)
so=optax.adam(1e-3); ss=so.init(eqx.filter(sde,eqx.is_array))
@eqx.filter_jit
def ss_step(sde,st,hg,tgt,sk):
    def lf(s): return jnp.mean((s(hg,t0=0.,t1=0.1,key=sk).ys[-1].reshape(ng,1)-tgt)**2)
    l,g=eqx.filter_value_and_grad(lf)(sde); u,ns=so.update(g,st,sde)
    return eqx.apply_updates(sde,u),ns,l
for ep in range(200):
    ti=ep%(T-1); sde,ss,l=ss_step(sde,ss,snaps[ti],jnp.array(temp[ti+1]).reshape(-1,1),jax.random.fold_in(k4,ep))
    if (ep+1)%50==0: print(f'  SDE ep {ep+1}: loss={l:.6f}')

print('Sampling 20 SDE trajectories...')
ts2=jnp.linspace(0,0.1*(T-1),T)
st=[np.array(sde(snaps[0],t0=0.,t1=float(ts2[-1]),key=jax.random.fold_in(k4,1000+i),saveat=diffrax.SaveAt(ts=ts2)).ys) for i in range(20)]
sig=np.exp(np.array(sde.diffusion.log_sigma))
print(f'  Sigma: [{sig.min():.4f},{sig.max():.4f}]')

fig,axes=plt.subplots(2,2,figsize=(14,12))
for i,(tf,idx) in enumerate(list(kti.items())[:4]):
    axes[0,0].plot(pt,temp[:,idx],'o-',color=C8[i],label=f'{tf} obs',lw=2)
    axes[0,0].plot(pt,op[:,idx],'s--',color=C8[i],label=f'{tf} pred',alpha=0.6)
axes[0,0].set(xlabel='Pseudotime',ylabel='Expression',title=f'A: Neural ODE (MSE={mse:.4f})')
axes[0,0].legend(fontsize=6,ncol=2); axes[0,0].grid(True,alpha=0.3)
for tr in st:
    axes[0,1].plot(np.array(ts2),[float(np.mean(np.abs(tr[t]))) for t in range(tr.shape[0])],alpha=0.2,color='#3498db',lw=1)
axes[0,1].plot(pt,[float(np.mean(np.abs(temp[t]))) for t in range(T)],'ko-',label='Observed',lw=2)
axes[0,1].set(xlabel='Time',ylabel='Mean |expr|',title='B: SDE Ensemble'); axes[0,1].legend()
res=np.abs(op[1:]-temp[1:]).mean(axis=0)
axes[1,0].hist(res,bins=50,color='#e74c3c',alpha=0.7)
axes[1,0].set(xlabel='Mean Abs Residual',ylabel='Count',title='C: ODE Error')
axes[1,1].hist(sig.flatten(),bins=50,color='#8e44ad',alpha=0.7)
axes[1,1].axvline(x=np.mean(sig),color='red',ls='--',label=f'mean={np.mean(sig):.4f}')
axes[1,1].set(xlabel='sigma',ylabel='Count',title='D: Learned Diffusion'); axes[1,1].legend()
fig.suptitle('Figure 6: Neural ODE/SDE Dynamics',fontsize=14,fontweight='bold')
fig.tight_layout(rect=[0,0,1,0.96])
fig.savefig(FIGP/'figure_06_dynamics.png',dpi=150,bbox_inches='tight'); plt.show(); print('Fig 6 done')

In [ ]:
# FIGURE 7: Perturbation Screen
kp1,kp2=jax.random.split(jax.random.PRNGKey(99))
pred_p=devograph.PerturbationPredictor(gene_dim=fd,hidden_dim=64,num_fates=3,conv_cls=hgx.UniGCNConv,num_layers=2,key=kp1)
te=jnp.broadcast_to(effects[:6,:,None],(6,n_genes,fd))
print('Training PerturbationPredictor...')
pred_p=devograph.train_perturbation_predictor(pred_p,hg,perturbations=masks[:6],targets=(te,fates[:6]),key=kp2,epochs=200,lr=1e-3)
for i,tf in enumerate(list(kti.keys())[6:]):
    ec2,fp2=devograph.in_silico_knockout(pred_p,hg,gene_idx=kti[tf])
    r=float(np.corrcoef(np.array(ec2).mean(-1),np.array(effects[6+i]))[0,1]) if np.std(np.array(effects[6+i]))>0 else 0
    print(f'  {tf} KO: r={r:.3f}')
tia=jnp.array([tgi[t] for t in tfs if t in tgi])
print(f'Screening {len(tia)} TFs...')
ac,af=devograph.perturbation_screen(pred_p,hg,tia)

fig,axes=plt.subplots(1,2,figsize=(14,5))
cn=np.array(ac.mean(axis=-1)); mg=np.abs(cn).mean(axis=1); t20=np.argsort(mg)[-20:]
stfs=[t for t in tfs if t in tgi]
im=axes[0].imshow(cn[t20,:50],aspect='auto',cmap='RdBu_r',vmin=-0.3,vmax=0.3)
axes[0].set_yticks(range(20)); axes[0].set_yticklabels([stfs[i] if i<len(stfs) else '' for i in t20],fontsize=6)
axes[0].set(xlabel='Gene (first 50)',title='A: Top 20 KO Effects'); plt.colorbar(im,ax=axes[0])
fn=np.array(af)
axes[1].scatter(fn[:,0],fn[:,1],c=fn[:,2],cmap='RdYlGn',s=10,alpha=0.5)
axes[1].set(xlabel='DF (cortical)',ylabel='VF (GE)',title=f'B: Fate Space ({len(tia)} KOs)')
axes[1].grid(True,alpha=0.3)
fig.suptitle('Figure 7: In-Silico Perturbation Screen',fontsize=14,fontweight='bold')
fig.tight_layout(rect=[0,0,1,0.96])
fig.savefig(FIGP/'figure_07_perturbation.png',dpi=150,bbox_inches='tight'); plt.show(); print('Fig 7 done')

In [ ]:
# FIGURE 8: Persistent Homology
try:
    rng=np.random.default_rng(42); si=rng.choice(n_genes,500,replace=False)
    si2=np.array(inc)[si]; ae=si2.sum(0)>=2
    shg=hgx.from_incidence(jnp.array(si2[:,ae]),node_features=feat[si])
    dgms=devograph.compute_persistence(shg,filtration='weight',max_dim=1)
    print(f'H0={len(dgms[0])}, H1={len(dgms[1])}')
    fig,axes=plt.subplots(1,2,figsize=(14,5))
    for d,dgm in enumerate(dgms):
        if len(dgm)>0:
            axes[0].scatter(dgm[:,0],dgm[:,1],c='#3498db' if d==0 else '#e74c3c',s=15,alpha=0.5,label=f'H{d} ({len(dgm)})')
    ap=np.concatenate([d for d in dgms if len(d)>0])
    axes[0].plot([ap.min(),ap.max()],[ap.min(),ap.max()],'k--',alpha=0.3)
    axes[0].set(xlabel='Birth',ylabel='Death',title='A: Persistence Diagram'); axes[0].legend()
    if len(dgms[0])>0:
        lt=dgms[0][:,1]-dgms[0][:,0]
        axes[1].hist(lt,bins=50,color='#3498db',alpha=0.7)
    axes[1].set(xlabel='Lifetime',ylabel='Count',title='B: H0 Lifetimes')
    fig.suptitle('Figure 8: Persistent Homology',fontsize=14,fontweight='bold')
    fig.tight_layout(rect=[0,0,1,0.96])
    fig.savefig(FIGP/'figure_08_persistence.png',dpi=150,bbox_inches='tight'); plt.show(); print('Fig 8 done')
except Exception as e: print(f'Persistence failed: {e}')

In [ ]:
print('\n'+'='*60+'\n  ALL FIGURES COMPLETE\n'+'='*60)
print(f'  Features: {fd}-dim (PPCA consensus)')
print(f'  Genes: {n_genes}, Regulons: {n_edges}')
print(f'  Figures saved to: {FIG}')
from IPython.display import display, Image
for f in sorted(FIGP.glob('*.png')):
    print(f'\n  {f.name}')
    display(Image(filename=str(f),width=800))